In [4]:
# Install required packages
!pip install -q umap-learn albumentations timm seaborn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os, sys, math, random, warnings, csv, json
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
import torch
import torchvision
import torchvision.transforms as T
from torchvision import models
import cv2

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

BASE = Path('./dataset')
TRAIN_DIR = BASE / 'train'
TEST_DIR  = BASE / 'test'
LABEL_CSV = BASE / 'label_mapping.csv'
WORK_DIR  = Path('./working')
WORK_DIR.mkdir(exist_ok=True)

matplotlib.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# Load label mapping
label_df = pd.read_csv(LABEL_CSV)
print(label_df.head())
print('Columns:', label_df.columns.tolist())

# Normalize column names
label_df.columns = [c.strip().lower() for c in label_df.columns]
# Expected columns: label_id, character_name (or similar)
id_col   = [c for c in label_df.columns if 'id' in c][0]
name_col = [c for c in label_df.columns if c != id_col][0]
label_df = label_df.rename(columns={id_col: 'label_id', name_col: 'character_name'})
id2name  = dict(zip(label_df['label_id'], label_df['character_name']))
print(f'Total classes in mapping: {len(label_df)}')

Device: cuda
   label_id                 class_name
0         0                    Acheron
1         1        Aerith_Gainsborough
2         2               Afuro_Terumi
3         3  Agano_(Kantai_Collection)
4         4           Agatsuma_Zenitsu
Columns: ['label_id', 'class_name']
Total classes in mapping: 630


# Section 1 — Class Distribution Histogram

**Why it matters:** Detecting class imbalance guides the decision to use `WeightedRandomSampler` or class-weighted loss — both critical for optimizing Macro F1.

In [6]:
try:
    # Collect all train images and parse label_id from folder or filename
    all_train = []
    for p in TRAIN_DIR.rglob('*'):
        if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
            all_train.append(p)

    # Try to infer label_id: folder name OR filename prefix (e.g. 0042_xxx.jpg)
    def infer_label(path):
        # Folder-based: TRAIN_DIR/<label_id>/image.jpg
        parent = path.parent.name
        if parent.isdigit():
            return int(parent)
        # Filename-based prefix: <label_id>_<rest>.jpg
        stem = path.stem.split('_')[0]
        if stem.isdigit():
            return int(stem)
        return -1

    labels_per_img = [infer_label(p) for p in all_train]
    valid_pairs = [(p, l) for p, l in zip(all_train, labels_per_img) if l >= 0]
    all_train_paths, all_train_labels = zip(*valid_pairs) if valid_pairs else ([], [])

    counts = Counter(all_train_labels)
    sorted_counts = sorted(counts.values(), reverse=True)
    cls_ids_sorted = [k for k, _ in sorted(counts.items(), key=lambda x: -x[1])]

    total_imgs = len(all_train_paths)
    mn, mx = min(sorted_counts), max(sorted_counts)
    mean_c = np.mean(sorted_counts)
    med_c  = np.median(sorted_counts)
    ratio  = mx / mn if mn > 0 else float('inf')

    fig, ax = plt.subplots(figsize=(16, 5))
    x = np.arange(len(sorted_counts))
    ax.bar(x, sorted_counts, color='steelblue', width=1.0, edgecolor='none', alpha=0.85)
    ax.axhline(mean_c, color='orange', lw=1.5, ls='--', label=f'Mean={mean_c:.1f}')
    ax.axhline(med_c,  color='green',  lw=1.5, ls='--', label=f'Median={med_c:.1f}')
    ax.axhline(mn,     color='red',    lw=1.5, ls='--', label=f'Min={mn}')
    ax.axhline(mx,     color='purple', lw=1.5, ls='--', label=f'Max={mx}')
    ax.set_xlabel('Class rank (sorted descending)')
    ax.set_ylabel('Image count')
    ax.set_title('Class Distribution Histogram (sorted descending)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec1_class_distribution.png', dpi=120)
    plt.show()

    use_wrs = ratio > 3
    print(f'Total images     : {total_imgs}')
    print(f'Min per class    : {mn}')
    print(f'Max per class    : {mx}')
    print(f'Mean per class   : {mean_c:.1f}')
    print(f'Median per class : {med_c:.1f}')
    print(f'Imbalance ratio  : {ratio:.1f}x')
    print(f'USE WeightedRandomSampler: {"YES" if use_wrs else "NO"}')

except Exception as e:
    print(f'[Section 1 ERROR] {e}')
    import traceback; traceback.print_exc()
    counts = Counter()
    all_train_paths, all_train_labels = [], []
    use_wrs = True
    mn, mx, mean_c, med_c, ratio = 0, 0, 0, 0, 0

Total images     : 93145
Min per class    : 1
Max per class    : 2
Mean per class   : 1.0
Median per class : 1.0
Imbalance ratio  : 2.0x
USE WeightedRandomSampler: NO


# Section 2 — Image Resolution and Aspect Ratio Distribution

**Why it matters:** Knowing the resolution spread confirms that 336×336 is a safe crop target and informs the `RandomResizedCrop` scale range.

In [ ]:
try:
    N_SAMPLE_RES = min(5000, len(all_train_paths))
    sampled_paths = random.sample(list(all_train_paths), N_SAMPLE_RES)

    widths, heights, aspects = [], [], []
    for p in sampled_paths:
        try:
            with Image.open(p) as im:
                w, h = im.size
            widths.append(w)
            heights.append(h)
            aspects.append(w / h)
        except:
            pass

    widths  = np.array(widths)
    heights = np.array(heights)
    aspects = np.array(aspects)
    shorter = np.minimum(widths, heights)

    def classify_ar(ar):
        if ar > 1.2:  return 'landscape'
        if ar < 0.83: return 'portrait'
        return 'square'

    cat_map = {'landscape': 'tab:blue', 'portrait': 'tab:orange', 'square': 'tab:green'}
    categories = [classify_ar(a) for a in aspects]
    colors_scatter = [cat_map[c] for c in categories]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Plot 1: scatter
    for cat, col in cat_map.items():
        mask = [c == cat for c in categories]
        axes[0].scatter(widths[mask], heights[mask], c=col, alpha=0.3, s=6, label=cat)
    axes[0].set_xlabel('Width (px)'); axes[0].set_ylabel('Height (px)')
    axes[0].set_title('Width vs Height')
    axes[0].legend(markerscale=3)

    # Plot 2: aspect ratio histogram
    axes[1].hist(aspects, bins=60, color='steelblue', edgecolor='none')
    axes[1].axvline(1.0, color='red', ls='--', label='Square')
    axes[1].set_xlabel('Aspect Ratio (W/H)'); axes[1].set_title('Aspect Ratio Histogram')
    axes[1].legend()

    # Plot 3: shorter side
    axes[2].hist(shorter, bins=60, color='salmon', edgecolor='none')
    axes[2].axvline(224, color='red', ls='--', label='224px')
    axes[2].axvline(336, color='orange', ls='--', label='336px')
    axes[2].set_xlabel('Shorter side (px)'); axes[2].set_title('Shorter Side Distribution')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec2_resolution.png', dpi=120)
    plt.show()

    pct_portrait  = 100 * categories.count('portrait')  / len(categories)
    pct_landscape = 100 * categories.count('landscape') / len(categories)
    pct_square    = 100 * categories.count('square')    / len(categories)
    pct_small     = 100 * np.mean(shorter < 224)
    med_w, med_h  = int(np.median(widths)), int(np.median(heights))
    ar_std = np.std(aspects)
    rec_scale = (0.7, 1.0) if ar_std > 0.25 else (0.8, 1.0)

    print(f'Portrait   : {pct_portrait:.1f}%')
    print(f'Landscape  : {pct_landscape:.1f}%')
    print(f'Square     : {pct_square:.1f}%')
    print(f'Median res : {med_w}x{med_h}')
    print(f'<224px     : {pct_small:.1f}% (risk of quality loss)')
    print(f'Recommended RandomResizedCrop scale: {rec_scale}')

except Exception as e:
    print(f'[Section 2 ERROR] {e}')
    import traceback; traceback.print_exc()
    rec_scale = (0.7, 1.0)

Portrait   : 0.0%
Landscape  : 0.0%
Square     : 100.0%
Median res : 224x224
<224px     : 0.0% (risk of quality loss)
Recommended RandomResizedCrop scale: (0.8, 1.0)


# Section 3 — Outlier and Easter Egg Detection (UMAP)

**Why it matters:** Identifying mislabeled or anomalous images helps clean training data, improving model generalization and Macro F1.

In [9]:
try:
    import umap

    N_UMAP = min(8000, len(all_train_paths))
    N_PER_CLASS = 12

    # Stratified sampling
    class_to_paths = defaultdict(list)
    for p, l in zip(all_train_paths, all_train_labels):
        class_to_paths[l].append(p)

    sampled_umap_paths, sampled_umap_labels = [], []
    for cls, paths in class_to_paths.items():
        k = min(N_PER_CLASS, len(paths))
        chosen = random.sample(paths, k)
        sampled_umap_paths.extend(chosen)
        sampled_umap_labels.extend([cls]*k)

    # Trim if over budget
    if len(sampled_umap_paths) > N_UMAP:
        idx = random.sample(range(len(sampled_umap_paths)), N_UMAP)
        sampled_umap_paths = [sampled_umap_paths[i] for i in idx]
        sampled_umap_labels= [sampled_umap_labels[i] for i in idx]

    # Build ResNet18 feature extractor
    backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    backbone.fc = torch.nn.Identity()
    backbone = backbone.to(DEVICE).eval()
    if DEVICE == 'cuda':
        backbone = backbone.half()

    preprocess = T.Compose([
        T.Resize(256),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])

    def get_embeddings(paths, batch_size=128):
        embs = []
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i+batch_size]
            imgs = []
            for p in batch_paths:
                try:
                    img = Image.open(p).convert('RGB')
                    imgs.append(preprocess(img))
                except:
                    imgs.append(torch.zeros(3, 224, 224))
            batch = torch.stack(imgs).to(DEVICE)
            if DEVICE == 'cuda':
                batch = batch.half()
            with torch.no_grad():
                out = backbone(batch).float().cpu().numpy()
            embs.append(out)
        return np.vstack(embs)

    print('Extracting embeddings...')
    embeddings = get_embeddings(sampled_umap_paths)
    print(f'Embeddings shape: {embeddings.shape}')

    print('Running UMAP...')
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=SEED)
    coords  = reducer.fit_transform(embeddings)

    umap_labels = np.array(sampled_umap_labels)
    n_cls = len(np.unique(umap_labels))
    cmap  = plt.cm.get_cmap('tab20', n_cls)
    label_to_idx = {l: i for i, l in enumerate(np.unique(umap_labels))}
    color_idx = [label_to_idx[l] for l in umap_labels]

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))

    axes[0].scatter(coords[:,0], coords[:,1], c=color_idx, cmap='nipy_spectral',
                    alpha=0.4, s=4, linewidths=0)
    axes[0].set_title('UMAP — colored by class (630 classes, no legend)')
    axes[0].set_xlabel('UMAP-1'); axes[0].set_ylabel('UMAP-2')

    # Outlier detection per class
    from sklearn.preprocessing import normalize
    emb_norm = normalize(embeddings)
    outlier_indices, outlier_distances = [], []
    class_list = np.unique(umap_labels)
    for cls in class_list:
        mask = umap_labels == cls
        cls_emb = emb_norm[mask]
        if len(cls_emb) < 3:
            continue
        mean_emb = cls_emb.mean(axis=0, keepdims=True)
        dists = 1 - (cls_emb @ mean_emb.T).flatten()
        threshold = dists.mean() + 2.5 * dists.std()
        idx_in_cls = np.where(mask)[0]
        for i, d in zip(idx_in_cls, dists):
            if d > threshold:
                outlier_indices.append(i)
                outlier_distances.append(d)

    outlier_indices = np.array(outlier_indices)
    outlier_distances = np.array(outlier_distances)

    axes[1].scatter(coords[:,0], coords[:,1], c='lightgray', alpha=0.2, s=4, linewidths=0)
    if len(outlier_indices) > 0:
        axes[1].scatter(coords[outlier_indices,0], coords[outlier_indices,1],
                       c='red', s=20, alpha=0.8, label=f'Outliers ({len(outlier_indices)})')
    axes[1].set_title('UMAP — outliers highlighted in red')
    axes[1].set_xlabel('UMAP-1'); axes[1].set_ylabel('UMAP-2')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec3_umap.png', dpi=120)
    plt.show()

    # Save outliers CSV
    outlier_rows = []
    for i, d in zip(outlier_indices, outlier_distances):
        outlier_rows.append({
            'path': str(sampled_umap_paths[i]),
            'label_id': sampled_umap_labels[i],
            'cosine_distance': float(d)
        })
    outlier_df = pd.DataFrame(outlier_rows).sort_values('cosine_distance', ascending=False)
    OUTLIER_CSV = WORK_DIR / 'outliers.csv'
    outlier_df.to_csv(OUTLIER_CSV, index=False)

    # Top 20 extreme outliers grid
    top20 = outlier_df.head(20)
    if len(top20) > 0:
        nrows, ncols = 4, 5
        fig, axes_grid = plt.subplots(nrows, ncols, figsize=(15, 12))
        for ax in axes_grid.flatten():
            ax.axis('off')
        for idx2, (_, row) in enumerate(top20.iterrows()):
            if idx2 >= nrows * ncols:
                break
            r, c = divmod(idx2, ncols)
            try:
                img = Image.open(row['path']).convert('RGB')
                img.thumbnail((150, 150))
                ax = axes_grid[r][c]
                ax.imshow(img)
                ax.set_title(f"cls={row['label_id']}\nd={row['cosine_distance']:.3f}",
                             fontsize=7)
                ax.axis('off')
            except:
                pass
        plt.suptitle('Top 20 Most Extreme Outliers', fontsize=12)
        plt.tight_layout()
        plt.savefig(WORK_DIR / 'sec3_outlier_grid.png', dpi=100)
        plt.show()

    outlier_class_counts = Counter(outlier_df['label_id'].tolist())
    top10_dense = outlier_class_counts.most_common(10)

    print(f'Total outliers: {len(outlier_df)}')
    print('Top 10 most outlier-dense classes:')
    for cls, cnt in top10_dense:
        print(f'  class {cls} ({id2name.get(cls, "?")}): {cnt} outliers')
    print(f'Outliers saved to: {OUTLIER_CSV}')

except Exception as e:
    print(f'[Section 3 ERROR] {e}')
    import traceback; traceback.print_exc()
    OUTLIER_CSV = WORK_DIR / 'outliers.csv'

Extracting embeddings...
Embeddings shape: (8000, 512)
Running UMAP...


[Section 3 ERROR] 'cosine_distance'


Traceback (most recent call last):
  File "C:\Users\harsh\AppData\Local\Temp\ipykernel_19764\1140620153.py", line 121, in <module>
    outlier_df = pd.DataFrame(outlier_rows).sort_values('cosine_distance', ascending=False)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\frame.py", line 8347, in sort_values
    k = self._get_label_or_level_values(by[0], axis=axis)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\pandas\core\generic.py", line 1776, in _get_label_or_level_values
    raise KeyError(key)
KeyError: 'cosine_distance'


# Section 4 — Style Separation Check

**Why it matters:** If styles (cel vs manga) are perfectly class-exclusive, style-balanced sampling is unnecessary; if mixed, it prevents style-induced overfitting.

In [ ]:
try:
    import colorsys

    def classify_style(path, sample_size=64):
        """Returns 'manga' or 'cel' based on HSV saturation and pixel std."""
        try:
            img = Image.open(path).convert('RGB').resize((sample_size, sample_size))
            arr = np.array(img, dtype=np.float32)
            # Convert to HSV
            img_hsv = img.convert('HSV') if hasattr(img, 'convert') else None
            # Use colorsys manually
            r, g, b = arr[:,:,0]/255, arr[:,:,1]/255, arr[:,:,2]/255
            # Vectorized HSV S channel
            maxc = np.maximum(np.maximum(r, g), b)
            minc = np.minimum(np.minimum(r, g), b)
            diff = maxc - minc
            s = np.where(maxc > 0, diff / maxc, 0.0)
            mean_sat = s.mean()
            std_pix  = arr.std()
            if mean_sat < 0.15 and std_pix > 60:
                return 'manga'
            return 'cel'
        except:
            return 'cel'

    # Sample up to 300 per class for speed
    STYLE_SAMPLE = 200
    class_style_data = defaultdict(lambda: {'manga': 0, 'cel': 0})
    style_paths_manga, style_paths_cel = [], []

    all_style_paths, all_style_labels = list(all_train_paths), list(all_train_labels)
    # Stratified sample
    cls2pairs = defaultdict(list)
    for p, l in zip(all_style_paths, all_style_labels):
        cls2pairs[l].append(p)

    style_sample_paths, style_sample_labels = [], []
    for cls, paths in cls2pairs.items():
        chosen = random.sample(paths, min(STYLE_SAMPLE, len(paths)))
        style_sample_paths.extend(chosen)
        style_sample_labels.extend([cls]*len(chosen))

    print(f'Classifying style for {len(style_sample_paths)} images...')
    styles = []
    for p in style_sample_paths:
        s = classify_style(p)
        styles.append(s)

    for p, lbl, s in zip(style_sample_paths, style_sample_labels, styles):
        class_style_data[lbl][s] += 1
        if s == 'manga' and len(style_paths_manga) < 10:
            style_paths_manga.append(p)
        elif s == 'cel' and len(style_paths_cel) < 10:
            style_paths_cel.append(p)

    # Per-class manga %
    manga_pcts = []
    for cls, d in class_style_data.items():
        total = d['manga'] + d['cel']
        if total > 0:
            manga_pcts.append((cls, 100*d['manga']/total))
    manga_pcts.sort(key=lambda x: x[1])

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    # Plot 1: histogram of per-class manga %
    pcts_vals = [p for _, p in manga_pcts]
    axes[0].hist(pcts_vals, bins=30, color='indigo', edgecolor='white')
    axes[0].set_xlabel('% Manga per class')
    axes[0].set_ylabel('Number of classes')
    axes[0].set_title('Per-class Manga Percentage Distribution')

    # Plot 2: top 30 and bottom 30 stacked bar
    show_cls = [x[0] for x in manga_pcts[:30]] + [x[0] for x in manga_pcts[-30:]]
    show_manga = [class_style_data[c]['manga'] for c in show_cls]
    show_cel   = [class_style_data[c]['cel']   for c in show_cls]
    x_bar = np.arange(len(show_cls))
    axes[1].bar(x_bar, show_manga, label='Manga', color='dimgray')
    axes[1].bar(x_bar, show_cel, bottom=show_manga, label='Cel', color='steelblue')
    axes[1].set_title('Top/Bottom 30 Classes by Manga %')
    axes[1].set_xlabel('Class index'); axes[1].set_ylabel('Image count')
    axes[1].legend()
    axes[1].axvline(30, color='red', ls='--', lw=1)

    # Plot 3: sample grid 2x3
    grid_paths = style_paths_cel[:3] + style_paths_manga[:3]
    grid_labels = ['CEL']*3 + ['MANGA']*3
    for i, (gp, gl) in enumerate(zip(grid_paths, grid_labels)):
        try:
            img = Image.open(gp).convert('RGB')
            img.thumbnail((200, 200))
            ax_g = axes[2] if i == 0 else None
        except:
            pass

    # Actually make a proper 2x3 grid in the 3rd slot
    axes[2].axis('off')
    axes[2].set_title('Style Sanity Check')

    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec4_style_main.png', dpi=120)
    plt.show()

    # Separate 2x3 figure for sample images
    fig2, axes2 = plt.subplots(2, 3, figsize=(12, 8))
    for i, (gp, gl) in enumerate(zip(grid_paths, grid_labels)):
        r, c = divmod(i, 3)
        try:
            img = Image.open(gp).convert('RGB')
            img.thumbnail((200, 200))
            axes2[r][c].imshow(img)
            axes2[r][c].set_title(gl, fontsize=10)
        except:
            pass
        axes2[r][c].axis('off')
    plt.suptitle('Style Sanity Check: CEL (top) vs MANGA (bottom)')
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec4_style_grid.png', dpi=120)
    plt.show()

    pct_exclusive = 100 * sum(1 for _, p in manga_pcts if p > 90 or p < 10) / len(manga_pcts)
    mean_manga_pct = np.mean(pcts_vals)
    style_balanced = mean_manga_pct > 20 and mean_manga_pct < 80

    print(f'Classes >90% one style: {pct_exclusive:.1f}%')
    print(f'Mean manga %: {mean_manga_pct:.1f}%')
    print(f'Style balanced sampling recommended: {"YES" if style_balanced else "NO"}')

except Exception as e:
    print(f'[Section 4 ERROR] {e}')
    import traceback; traceback.print_exc()
    style_balanced = False
    class_style_data = {}

# Section 5 — Per-Class Count Extremes and Oversampling Decision

**Why it matters:** Small classes can vanish in training batches — knowing which need oversampling directly creates the copy-pasteable config for the training script.

In [ ]:
try:
    sorted_classes = sorted(counts.items(), key=lambda x: x[1])
    bottom20 = sorted_classes[:20]
    top20_cls = sorted_classes[-20:]

    # Horizontal bar chart
    fig, ax = plt.subplots(figsize=(10, 8))
    cls_names_b20 = [id2name.get(c, str(c)) for c, _ in bottom20]
    cls_counts_b20 = [n for _, n in bottom20]
    bar_colors = ['red' if n < 50 else ('orange' if n <= 100 else 'green') for n in cls_counts_b20]

    bars = ax.barh(range(len(bottom20)), cls_counts_b20, color=bar_colors, edgecolor='none')
    ax.set_yticks(range(len(bottom20)))
    ax.set_yticklabels([f'{id2name.get(c, str(c))[:25]} ({c})' for c, _ in bottom20], fontsize=8)
    ax.set_xlabel('Image count')
    ax.set_title('Bottom 20 Classes by Image Count')
    ax.axvline(50, color='red', ls='--', lw=1, label='<50 (red)')
    ax.axvline(100, color='orange', ls='--', lw=1, label='<100 (orange)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec5_bottom20_bar.png', dpi=120)
    plt.show()

    # Thumbnail grid 4x5 for bottom 20
    fig, axes_g = plt.subplots(4, 5, figsize=(15, 12))
    for ax in axes_g.flatten(): ax.axis('off')
    for idx3, (cls, _) in enumerate(bottom20):
        r, c = divmod(idx3, 5)
        paths_for_cls = [p for p, l in zip(all_train_paths, all_train_labels) if l == cls]
        if paths_for_cls:
            try:
                img = Image.open(random.choice(paths_for_cls)).convert('RGB')
                img.thumbnail((150, 150))
                axes_g[r][c].imshow(img)
                n_img = counts[cls]
                axes_g[r][c].set_title(f'cls={cls}\nn={n_img}', fontsize=7)
            except:
                pass
    plt.suptitle('Bottom 20 Classes — Sample Image')
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec5_bottom20_grid.png', dpi=100)
    plt.show()

    # Oversample map
    oversample_map = {}
    for cls, n in counts.items():
        if n < 100:
            factor = math.ceil(100 / n)
            oversample_map[cls] = factor

    print('Classes with <50 images:')
    for cls, n in sorted_classes:
        if n < 50:
            print(f'  cls {cls} ({id2name.get(cls, "?")[:30]}): {n} images, oversample x{math.ceil(100/n)}')

    print(f'\nTotal classes needing oversampling (<100 imgs): {len(oversample_map)}')
    print(f'oversample_map (first 10): {dict(list(oversample_map.items())[:10])}')

except Exception as e:
    print(f'[Section 5 ERROR] {e}')
    import traceback; traceback.print_exc()
    oversample_map = {}

# Section 6 — Augmentation Sanity Grid

**Why it matters:** Visually confirming augmentations preserve character identity ensures the model learns from valid transformations rather than corrupted inputs.

In [ ]:
try:
    import albumentations as A
    from scipy.stats import entropy as scipy_entropy

    def img_entropy(pil_img):
        arr = np.array(pil_img.convert('L'))
        hist, _ = np.histogram(arr.flatten(), bins=256, range=(0,255))
        hist = hist / hist.sum()
        return scipy_entropy(hist + 1e-12)

    # Pick 4 images: high-count cel, high-count manga, low-count cel, low-count manga
    style_labels_map = {}
    for p, l, s in zip(style_sample_paths if 'style_sample_paths' in dir() else [], 
                       style_sample_labels if 'style_sample_labels' in dir() else [], 
                       styles if 'styles' in dir() else []):
        style_labels_map[str(p)] = (l, s)

    top_classes = [k for k, _ in sorted(counts.items(), key=lambda x: -x[1])[:50]]
    bot_classes = [k for k, _ in sorted(counts.items(), key=lambda x:  x[1])[:50]]

    def find_img(class_list, style):
        for cls in class_list:
            for p in random.sample(cls2pairs.get(cls, [])[:20], 
                                   min(10, len(cls2pairs.get(cls, [])))):
                s = classify_style(p)
                if s == style:
                    return p, cls, style
        # fallback
        cls = class_list[0]
        paths = cls2pairs.get(cls, [])
        if paths:
            return random.choice(paths), cls, 'unknown'
        return None, None, None

    quad = [
        find_img(top_classes, 'cel'),
        find_img(top_classes, 'manga'),
        find_img(bot_classes, 'cel'),
        find_img(bot_classes, 'manga'),
    ]
    quad_labels = ['high-cnt cel', 'high-cnt manga', 'low-cnt cel', 'low-cnt manga']

    # Define augmentations
    aug_names = [
        'Original',
        'RRC(336)',
        'ColorJitter',
        'Grayscale',
        'Posterize',
        'GaussianBlur',
        'RandAugment',
    ]

    def apply_aug(pil_img, aug_idx):
        if aug_idx == 0:
            return pil_img.resize((336, 336))
        elif aug_idx == 1:
            return T.Compose([
                T.RandomResizedCrop(336, scale=(0.7, 1.0)),
            ])(pil_img)
        elif aug_idx == 2:
            return T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1)(pil_img)
        elif aug_idx == 3:
            return T.RandomGrayscale(p=1.0)(pil_img)
        elif aug_idx == 4:
            return T.RandomPosterize(bits=4, p=1.0)(pil_img)
        elif aug_idx == 5:
            return T.GaussianBlur(kernel_size=3)(pil_img)
        elif aug_idx == 6:
            return T.RandAugment(num_ops=2, magnitude=9)(pil_img)

    n_augs = len(aug_names)
    n_rows = 4
    fig, axes_a = plt.subplots(n_rows, n_augs, figsize=(n_augs*2.5, n_rows*2.8))

    aug_issues = []
    for row_idx, ((path, cls, style), quad_lbl) in enumerate(zip(quad, quad_labels)):
        if path is None:
            for c in range(n_augs):
                axes_a[row_idx][c].axis('off')
            continue
        try:
            orig_img = Image.open(path).convert('RGB')
            orig_entropy = img_entropy(orig_img)
            for aug_idx in range(n_augs):
                try:
                    aug_img = apply_aug(orig_img.copy(), aug_idx)
                    if aug_img.mode != 'RGB':
                        aug_img = aug_img.convert('RGB')
                    aug_img_disp = aug_img.resize((200, 200))
                    ax = axes_a[row_idx][aug_idx]
                    ax.imshow(aug_img_disp)
                    ax.axis('off')
                    if row_idx == 0:
                        ax.set_title(aug_names[aug_idx], fontsize=9, fontweight='bold')
                    if aug_idx == 0:
                        ax.set_ylabel(f'{quad_lbl}\ncls={cls}', fontsize=7)
                    # Check entropy drop
                    if aug_idx > 0:
                        aug_ent = img_entropy(aug_img)
                        if orig_entropy > 0 and (orig_entropy - aug_ent) / orig_entropy > 0.5:
                            issue = f'Aug "{aug_names[aug_idx]}" drops entropy >{50}% on {quad_lbl}'
                            if issue not in aug_issues:
                                aug_issues.append(issue)
                except Exception as ae:
                    axes_a[row_idx][aug_idx].axis('off')
                    axes_a[row_idx][aug_idx].set_title(f'ERR', fontsize=7)
        except Exception as re:
            print(f'Row {row_idx} error: {re}')

    plt.suptitle('Augmentation Sanity Grid (4 source images × 7 augmentations)', fontsize=12)
    plt.tight_layout()
    plt.savefig(WORK_DIR / 'sec6_aug_grid.png', dpi=100)
    plt.show()

    print('Augmentation issues detected:')
    if aug_issues:
        for iss in aug_issues:
            print(f'  WARNING: {iss}')
    else:
        print('  None — all augmentations preserve entropy within 50% threshold.')

except Exception as e:
    print(f'[Section 6 ERROR] {e}')
    import traceback; traceback.print_exc()
    aug_issues = []

# PREPROCESSING DECISIONS SUMMARY

In [ ]:
PREPROCESSING_CONFIG = {
    'use_weighted_sampler':    use_wrs     if 'use_wrs'         in dir() else True,
    'recommended_crop_scale':  rec_scale   if 'rec_scale'       in dir() else (0.7, 1.0),
    'outlier_csv_path':        str(OUTLIER_CSV) if 'OUTLIER_CSV' in dir() else '/kaggle/working/outliers.csv',
    'style_balanced_sampling': style_balanced if 'style_balanced' in dir() else False,
    'oversample_map':          oversample_map if 'oversample_map' in dir() else {},
    'augmentation_issues':     aug_issues  if 'aug_issues'      in dir() else [],
}

print('='*60)
print('PREPROCESSING_CONFIG = {')
for k, v in PREPROCESSING_CONFIG.items():
    print(f'    "{k}": {repr(v)},')
print('}')
print('='*60)